# AutoML v2 - B-4064A com regimes reais de 2025

Este notebook usa `Dados-novos/Histórico falha B-4064A.xlsx` para separar janelas normais, indisponibilidade e restrição. A análise evita calibrar threshold em períodos com reparo/restrição e compara três caminhos:

- threshold por regime sobre o score existente;
- retreino com janelas normais reais de 2024 + 2025;
- retreino com features causais de tendência/variação.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists()),
    Path.cwd(),
)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from transpetro_modelos.analysis.b4064a_regime_recalibration import (
    ExperimentConfig,
    MANUAL_WINDOWS,
    availability_intervals,
    evaluate_scores,
    load_failure_history,
    load_pre_split_data,
    score_existing_by_regime,
    train_and_score_dense,
    window_sensor_summary,
)


In [ ]:
# ==============================
# Configuracao
# ==============================
HISTORY_PATH = PROJECT_ROOT / 'Dados-novos' / 'Histórico falha B-4064A.xlsx'

# Opcional: score existente do AutoML v2 salvo/cacheado.
# Se estiver vazio, pule a celula de threshold por regime ou carregue via ClearML manualmente.
EXISTING_SCORES_PATH = Path('/home/chico/.clearml/cache/storage_manager/global/d14b8460acd6a88ca4830f44f0a38f84.best_full_scores.csv.gz')

THRESHOLD_PERCENTILE = 99.0
PERSISTENCE_K = 4
PERSISTENCE_N = 6

# Retreino local. Reduza epochs para teste rapido, aumente para analise final.
TRAIN_CONFIG = ExperimentConfig(
    preset='thermal_ma',
    threshold_percentile=99.0,
    persistence_k=PERSISTENCE_K,
    persistence_n=PERSISTENCE_N,
    epochs=60,
    patience=8,
    device='cpu',
)


In [ ]:
history = load_failure_history(HISTORY_PATH)
intervals = availability_intervals(history)

display(intervals[['start', 'end', 'status', 'comments']])
print('Janelas manuais usadas na analise:')
for name, (start, end) in MANUAL_WINDOWS.items():
    print(f'{name:28s}: {start} -> {end}')


In [ ]:
df_pre = load_pre_split_data()
print(df_pre.shape, df_pre.index.min(), '->', df_pre.index.max())

sensor_summary = window_sensor_summary(df_pre)
display(
    sensor_summary
    .query("sensor in ['Temperatura Bomba LNA', 'Vibração Bomba LNA', 'Temperatura Motor LNA']")
    .pivot(index='window', columns='sensor', values='median')
    .round(3)
)


## 1. Threshold por regime no score existente

Mantém um threshold histórico para a falha de 2024 e calibra outro threshold para o normal pós-reparo de 2025. Isso testa rapidamente se o problema principal é drift de regime.


In [ ]:
def load_existing_scores(path):
    scores = pd.read_csv(path)
    scores['Timestamp'] = pd.to_datetime(scores['Timestamp'])
    return scores.set_index('Timestamp').sort_index()[['reconstruction_error', 'is_anomaly']]

existing_scores = load_existing_scores(EXISTING_SCORES_PATH)
regime_summary, regime_thresholds = score_existing_by_regime(
    existing_scores,
    percentile=THRESHOLD_PERCENTILE,
    k=PERSISTENCE_K,
    n=PERSISTENCE_N,
)
print(regime_thresholds)
display(regime_summary)


In [ ]:
ax = existing_scores['reconstruction_error'].plot(figsize=(14, 4), linewidth=0.8, alpha=0.7)
ax.axhline(regime_thresholds['threshold_2024'], color='tab:red', linestyle='--', label='threshold 2024')
ax.axhline(regime_thresholds['threshold_2025'], color='tab:green', linestyle='--', label='threshold pos-reparo')
for name in ['falha_2024_15d', 'restricao_selo_2025', 'normal_2025_pos_restricao']:
    start, end = MANUAL_WINDOWS[name]
    ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.12, label=name)
ax.set_title('Score existente com thresholds por regime')
ax.set_ylabel('Score de anomalia')
ax.legend(loc='upper left', fontsize=8)
plt.show()


## 2. Retreino com normais reais

Treina um autoencoder denso usando `normal_2024`, `normal_2025_jan_mar` e `normal_2025_pos_restricao`. A restrição de abril/2025 fica apenas como avaliação.


In [ ]:
retrained_scores, retrained_threshold = train_and_score_dense(
    df_pre,
    TRAIN_CONFIG,
    use_trend_features=False,
)
retrained_summary = evaluate_scores(
    retrained_scores,
    retrained_threshold,
    k=PERSISTENCE_K,
    n=PERSISTENCE_N,
)
print('threshold retreinado:', retrained_threshold)
display(retrained_summary)


## 3. Retreino com features de tendência

Adiciona features causais de diferença, média móvel curta e z-score móvel longo para temperaturas/vibração, tentando capturar degradação relativa em vez de temperatura absoluta.


In [ ]:
trend_scores, trend_threshold = train_and_score_dense(
    df_pre,
    TRAIN_CONFIG,
    use_trend_features=True,
)
trend_summary = evaluate_scores(
    trend_scores,
    trend_threshold,
    k=PERSISTENCE_K,
    n=PERSISTENCE_N,
)
print('threshold trend:', trend_threshold)
display(trend_summary)


In [ ]:
combined = pd.concat([
    regime_summary.assign(approach='existing_score_regime_threshold'),
    retrained_summary.assign(approach='retrained_dense_manual_normals'),
    trend_summary.assign(approach='retrained_dense_trend_features'),
], ignore_index=True)

cols = ['approach', 'window', 'n_samples', 'n_alerts', 'alert_rate', 'score_median', 'score_p95', 'score_p99', 'threshold']
display(combined[cols].sort_values(['window', 'approach']).style.format({
    'alert_rate': '{:.2%}',
    'score_median': '{:.4f}',
    'score_p95': '{:.4f}',
    'score_p99': '{:.4f}',
    'threshold': '{:.4f}',
}))
